Parcial 2, Labo 2

Nicolas Olivares

Ejercicio 1

In [ ]:
class EnergiaInsuficienteError(Exception):
    pass

def protocolo_enlace(reintentos):
    def decorador(func):
        def wrapper(*args, **kwargs):
            print("[Base] estableciendo enlace con el dispositivo...")
            
            for intento in range(reintentos):
                try:
                    return func(*args, **kwargs)
                except EnergiaInsuficienteError:
                    print(f"[Base] fallo la transmision, te quedan {reintentos - intento - 1}")
            print("[Base] no se pudo establecer el enlace")
        return wrapper
    return decorador

from abc import ABC, abstractmethod

class DispositivoEspecial(ABC):
    @abstractmethod
    def transmitir_datos(self):
        pass

    @abstractmethod
    def estado_bateria(self):
        pass

    @classmethod
    def dispositivos_activos(cls):
        return cls.dispositivos_activos

In [ ]:
import random

class SateliteOrbital(DispositivoEspecial):

    def __init__(self, codigo_satelite):
        self.codigo_satelite = codigo_satelite
        self.__nivel_energia = 100
        self.__modo_ahorro = False

    @protocolo_enlace(reintentos=3)
    def transmitir_datos(self):
        consumo = random.randint(10, 40)
        self.consumir_energia(consumo)

    def obtener_energia(self):
        return self.__nivel_energia
    
    def consumir_energia(self, cantidad):
        self.__nivel_energia -= cantidad
        if self.__nivel_energia <= 20:
            self.__modo_ahorro = True
        if self.__nivel_energia < 0:
            self.__nivel_energia = 0
            raise EnergiaInsuficienteError("energía insuficiente")
        
    def estado_bateria(self):
        return f"satelite {self.codigo_satelite}: energia {self.__nivel_energia}, modo ahorro: {self.__modo_ahorro}"

In [ ]:
class SondaTerrestre(DispositivoEspecial):

    def __init__(self, codigo_sonda):
        self.codigo_sonda = codigo_sonda
        self.__bateria_litio = 50
        self.__modo_ahorro = False

    @protocolo_enlace(reintentos=1)
    def transmitir_datos(self):
        consumo = random.randint(5, 25)
        self.consumir_energia(consumo)

    def obtener_energia(self):
        return self.__bateria_litio
    
    def consumir_energia(self, cantidad):
        self.__bateria_litio -= cantidad
        if self.__bateria_litio <= 10:
            self.__modo_ahorro = True
        if self.__bateria_litio < 0:
            self.__bateria_litio = 0
            raise EnergiaInsuficienteError("energia insuficiente")
        
    def estado_bateria(self):
        return f"sonda {self.codigo_sonda}: bateria {self.__bateria_litio}, modo ahorro: {self.__modo_ahorro}"

In [58]:
satelite = SateliteOrbital("UwU")
satelite.transmitir_datos()
print(satelite.estado_bateria())

sonda = SondaTerrestre("7v7")
sonda.transmitir_datos()
print(sonda.estado_bateria())

dispositivos = [satelite, sonda]
for dispositivo in dispositivos:
    for i in range(3):
        dispositivo.transmitir_datos()
    print(dispositivo.estado_bateria())

[Base] estableciendo enlace con el dispositivo...
satelite UwU: energia 79, modo ahorro: False
[Base] estableciendo enlace con el dispositivo...
sonda 7v7: bateria 31, modo ahorro: False
[Base] estableciendo enlace con el dispositivo...
[Base] estableciendo enlace con el dispositivo...
[Base] estableciendo enlace con el dispositivo...
[Base] fallo la transmision, te quedan 2
[Base] fallo la transmision, te quedan 1
[Base] fallo la transmision, te quedan 0
[Base] no se pudo establecer el enlace
satelite UwU: energia 0, modo ahorro: True
[Base] estableciendo enlace con el dispositivo...
[Base] estableciendo enlace con el dispositivo...
[Base] fallo la transmision, te quedan 0
[Base] no se pudo establecer el enlace
[Base] estableciendo enlace con el dispositivo...
[Base] fallo la transmision, te quedan 0
[Base] no se pudo establecer el enlace
sonda 7v7: bateria 0, modo ahorro: True


Ejercicio 2

In [63]:
def generar_lotes_urls(rango_inicio, rango_fin):
    for id in range(rango_inicio, rango_fin):
        yield f" https://pokeapi.co/api/v2/pokemon/{id}"


import requests

def extraer_datos_pokemon(url):
    try:
        response = requests.get(url, timeout=2)
        response.raise_for_status()
    except requests.exceptions.Timeout:
        return None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            return None
        
    datos = response.json()
    name = datos['name']
    height = datos['height'] / 10.0
    weight = datos['weight'] / 10.0

    return {'name': name, 'height': height, 'weight': weight}


import threading
import queue

cola_descargas = queue.Queue()
pokedex_segura = []
cuando_pokedex = threading.Lock()

def Productor():
    for url in generar_lotes_urls(1, 60):
        cola_descargas.put(url)

def Consumidor():
    while True:
        url = cola_descargas.get()
        datos_pokemon = extraer_datos_pokemon(url)
        if datos_pokemon is not None:
            with cuando_pokedex:
                pokedex_segura.append(datos_pokemon)
        cola_descargas.task_done()

productor = threading.Thread(target=Productor)
productor.start()
consumidores = []

for i in range(6):
    c = threading.Thread(target=Consumidor, daemon=True)
    c.start()
    consumidores.append(c)
cola_descargas.join()

print(f"Cantidad de pokemons registrados: {len(pokedex_segura)}")
if pokedex_segura:
    print(f"Primer pokemon: {pokedex_segura[0]['name']}")
    print(f"Ultimo pokemon: {pokedex_segura[-1]['name']}")

Cantidad de pokemons registrados: 59
Primer pokemon: ivysaur
Ultimo pokemon: arcanine
